# 2. Protocolo de validação temporal

Este notebook define e valida os protocolos de validação temporal utilizados
no experimento de comparação dos modelos de previsão da taxa de evasão.

Serão avaliadas duas estratégias:

- Expanding Window
- Rolling Window

Os dois protocolos utilizarão os mesmos períodos de teste, diferenciando-se
apenas pela quantidade de histórico utilizada no treinamento.

## 2.1 Configuração

In [1]:
from pathlib import Path
import sys
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd()

while (
    PROJECT_ROOT != PROJECT_ROOT.parent
    and not (PROJECT_ROOT / "src").is_dir()
):
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Raiz do projeto: {PROJECT_ROOT}")

Raiz do projeto: /home/sara/Documentos/tcc-evasao-ensino-superior


In [3]:
from src.validation import criar_folds_temporais

## 2.2 Carregamento da base processada

In [4]:
CAMINHO_BASE_PROCESSADA = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "base_modelo_uf.csv"
)

df_modelo_base = pd.read_csv(
    CAMINHO_BASE_PROCESSADA
)

print(f"Arquivo: {CAMINHO_BASE_PROCESSADA}")
print(f"Dimensões: {df_modelo_base.shape}")

Arquivo: /home/sara/Documentos/tcc-evasao-ensino-superior/data/processed/base_modelo_uf.csv
Dimensões: (189, 10)


In [5]:
df_modelo_base.head()

,ano_fluxo,uf,evasao_t,conclusao_t,retencao_t,permanencia_t,evasao_t_1,conclusao_t_1,retencao_t_1,permanencia_t_1
0,2017-2018,Acre,13.9,76.2,23.4,79.9,13.9,77.6,25.4,80.2
1,2018-2019,Acre,13.2,75.3,22.7,81.3,13.9,76.2,23.4,79.9
2,2019-2020,Acre,11.5,68.2,36.3,83.9,13.2,75.3,22.7,81.3
3,2020-2021,Acre,14.3,59.8,40.2,80.8,11.5,68.2,36.3,83.9
4,2021-2022,Acre,15.2,68.4,33.3,78.4,14.3,59.8,40.2,80.8


In [6]:
df_modelo_base.columns.tolist()

['ano_fluxo',
 'uf',
 'evasao_t',
 'conclusao_t',
 'retencao_t',
 'permanencia_t',
 'evasao_t_1',
 'conclusao_t_1',
 'retencao_t_1',
 'permanencia_t_1']

## 2.3 Preparação temporal

In [7]:
df_modelo_base["ano_inicio"] = (
    df_modelo_base["ano_fluxo"]
    .str[:4]
    .astype(int)
)

In [8]:
periodos = sorted(
    df_modelo_base["ano_inicio"].unique()
)

print("Períodos disponíveis:")
print(periodos)

Períodos disponíveis:
[np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


In [9]:
print("Quantidade de períodos:", len(periodos))
print("Quantidade de UFs:", df_modelo_base["uf"].nunique())

Quantidade de períodos: 7
Quantidade de UFs: 27


## 2.4 Protocolo Expanding Window

In [10]:
folds_expanding = criar_folds_temporais(
    df_modelo_base,
    estrategia="expanding",
    tamanho_treino=4
)

In [11]:
pd.DataFrame(folds_expanding)

,fold,periodos_treino,periodo_teste,n_treino,n_teste,ufs_treino,ufs_teste
0,1,"[2017, 2018, 2019, 2020]",2021,108,27,27,27
1,2,"[2017, 2018, 2019, 2020, 2021]",2022,135,27,27,27
2,3,"[2017, 2018, 2019, 2020, 2021, 2022]",2023,162,27,27,27


## 2.5 Protocolo Rolling Window

In [12]:
folds_rolling = criar_folds_temporais(
    df_modelo_base,
    estrategia="rolling",
    tamanho_treino=4
)

In [13]:
pd.DataFrame(folds_rolling)

,fold,periodos_treino,periodo_teste,n_treino,n_teste,ufs_treino,ufs_teste
0,1,"[2017, 2018, 2019, 2020]",2021,108,27,27,27
1,2,"[2018, 2019, 2020, 2021]",2022,108,27,27,27
2,3,"[2019, 2020, 2021, 2022]",2023,108,27,27,27


## 2.6 Validação dos protocolos

In [14]:
for folds, nome in [
    (folds_expanding, "Expanding"),
    (folds_rolling, "Rolling")
]:
    for fold in folds:

        assert fold["n_teste"] == 27
        assert fold["ufs_teste"] == 27
        assert fold["ufs_treino"] == 27

    print(
        f"{nome}: protocolo estrutural validado."
    )

Expanding: protocolo estrutural validado.
Rolling: protocolo estrutural validado.


In [15]:
for folds, nome in [
    (folds_expanding, "Expanding"),
    (folds_rolling, "Rolling")
]:

    for fold in folds:

        maior_periodo_treino = max(
            fold["periodos_treino"]
        )

        periodo_teste = fold["periodo_teste"]

        assert maior_periodo_treino < periodo_teste

    print(
        f"{nome}: nenhum período futuro "
        "foi incluído no treinamento."
    )

Expanding: nenhum período futuro foi incluído no treinamento.
Rolling: nenhum período futuro foi incluído no treinamento.


In [16]:
testes_expanding = [
    fold["periodo_teste"]
    for fold in folds_expanding
]

testes_rolling = [
    fold["periodo_teste"]
    for fold in folds_rolling
]

assert testes_expanding == testes_rolling

print(
    "Os dois protocolos utilizam os mesmos "
    "períodos de teste:"
)

print(testes_expanding)

Os dois protocolos utilizam os mesmos períodos de teste:
[np.int64(2021), np.int64(2022), np.int64(2023)]


In [17]:
ufs_esperadas = set(
    df_modelo_base["uf"].unique()
)

for folds, nome in [
    (folds_expanding, "Expanding"),
    (folds_rolling, "Rolling")
]:

    for fold in folds:

        periodo_teste = fold["periodo_teste"]

        ufs_teste = set(
            df_modelo_base.loc[
                df_modelo_base["ano_inicio"] == periodo_teste,
                "uf"
            ]
        )

        assert ufs_teste == ufs_esperadas

    print(
        f"{nome}: mesmas 27 UFs em todos os períodos de teste."
    )

Expanding: mesmas 27 UFs em todos os períodos de teste.
Rolling: mesmas 27 UFs em todos os períodos de teste.


## 2.7 Síntese do protocolo

O experimento utilizará validação temporal com três períodos de teste:
2021–2022, 2022–2023 e 2023–2024.

No protocolo Expanding Window, todo o histórico disponível até o período
anterior ao teste é utilizado no treinamento.

No protocolo Rolling Window, são utilizados os quatro períodos imediatamente
anteriores ao período de teste.

Os dois protocolos utilizam exatamente os mesmos períodos e as mesmas
27 unidades federativas nos conjuntos de teste.

Nenhum período futuro em relação ao teste é utilizado no treinamento.